In [1]:
from surprise import Dataset, Reader, SVD
import pandas as pd

from surprise.accuracy import rmse, mae
import numpy as np
import math
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [1]:
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares

c:\Users\PC\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


# Đọc file ratings.dat
ratings_df = pd.read_csv('ratings.dat', sep='::', engine='python', 
                      names=['UserId', 'MovieId', 'Rating', 'Timestamp'])

# Đọc file tags.dat
tags_df = pd.read_csv('tags.dat', sep='::', engine='python', 
                   names=['UserId', 'MovieId', 'Tag', 'Timestamp'])

# Đọc file movies.dat
movies_df = pd.read_csv('movies.dat', sep='::', engine='python', 
                     names=['MovieId', 'Title', 'Genres'])

In [3]:
# Loại bỏ các hàng có dữ liệu thiếu
ratings_df.dropna(inplace=True)
tags_df.dropna(inplace=True)
movies_df.dropna(inplace=True)

In [4]:
from sklearn.model_selection import train_test_split

# Load
df = ratings_df[["UserId", "MovieId", "Rating"]].copy()

# 80% train, 20% temp
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)

# 10% val, 10% test
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(len(train_df), len(val_df), len(test_df))

8000043 1000005 1000006


In [5]:
from surprise import Dataset, Reader, SVD

reader = Reader(rating_scale=(0.5, 5.0))

train_data = Dataset.load_from_df(
    train_df[["UserId", "MovieId", "Rating"]],
    reader
)

trainset = train_data.build_full_trainset()

model_SVD = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

model_SVD.fit(trainset)

In [7]:


with open("data/svd_model2.pkl", "wb") as f:
    pickle.dump(model_SVD, f)

print("Đã lưu model vào data/svd_model2.pkl")

Đã lưu model vào data/svd_model2.pkl


In [6]:
from surprise import NMF

# NMF trong surprise dùng tên tham số hơi khác SVD một chút
nmf_model = NMF(
    n_factors=50,      # Giữ nguyên số latent factors
    n_epochs=20,              # Thay reg_all → reg
    random_state=42    # Giữ seed để tái lập kết quả
)

print("🔄 Đang train NMF...")
nmf_model.fit(trainset)
print("✅ NMF huấn luyện hoàn tất!")

🔄 Đang train NMF...
✅ NMF huấn luyện hoàn tất!


In [7]:
from surprise import SVDpp

model_SVDpp = SVDpp(
    n_factors=40,
    n_epochs=12,
    lr_all=0.007,
    reg_all=0.02,
    random_state=42
)
model_SVDpp.fit(trainset)

KeyboardInterrupt: 

ALS


In [10]:


# 1. Tạo mapping ID gốc -> index liên tục
all_users = sorted(pd.concat([train_df['UserId'], val_df['UserId'], test_df['UserId']]).unique())
all_movies = sorted(pd.concat([train_df['MovieId'], val_df['MovieId'], test_df['MovieId']]).unique())

user2idx = {uid: i for i, uid in enumerate(all_users)}
movie2idx = {mid: i for i, mid in enumerate(all_movies)}
idx2user = {i: uid for uid, i in user2idx.items()}
idx2movie = {i: mid for mid, i in movie2idx.items()}

n_users = len(all_users)
n_items = len(all_movies)

# 2. Hàm chuyển DataFrame sang sparse matrix
def df_to_sparse(df, u_map, i_map, n_u, n_i):
    rows = df['MovieId'].map(i_map).values  # items ở hàng
    cols = df['UserId'].map(u_map).values   # users ở cột
    data = df['Rating'].values              # giá trị rating làm trọng số
    return sp.csr_matrix((data, (rows, cols)), shape=(n_i, n_u))

train_sparse = df_to_sparse(train_df, user2idx, movie2idx, n_users, n_items)
val_sparse   = df_to_sparse(val_df, user2idx, movie2idx, n_users, n_items)
test_sparse  = df_to_sparse(test_df, user2idx, movie2idx, n_users, n_items)

In [11]:
# Cấu hình chuẩn cho explicit feedback
model_als = AlternatingLeastSquares(
    factors=50,            # Số latent factors (tương đương n_factors của SVD)
    iterations=15,         # Số vòng lặp ALS
    regularization=0.01,   # Điều chuẩn L2
    calculate_training_loss=False,
    num_threads=0          # 0 = dùng tất cả CPU cores
)

# ALS trong implicit yêu cầu ma trận (items x users)
print("Đang train ALS...")
model_als.fit(train_sparse)
print("✅ Train xong!")

c:\Users\PC\AppData\Local\Programs\Python\Python310\lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 20 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Đang train ALS...


100%|██████████| 15/15 [00:06<00:00,  2.37it/s]

✅ Train xong!


In [19]:
class ALSWrapper:
    def __init__(self, als_model, u_map, i_map):
        self.model = als_model
        self.u_map = u_map
        self.i_map = i_map
        
    class Pred:
        def __init__(self, est): self.est = est
            
    def predict(self, uid, iid):
        u_idx = self.u_map.get(uid)
        i_idx = self.i_map.get(iid)
        if u_idx is None or i_idx is None:
            return self.Pred(0.0)  # Cold-start fallback
        # Score = dot product của latent vectors
        score = self.model.user_factors[u_idx] @ self.model.item_factors[i_idx]
        return self.Pred(score)

# Khởi tạo wrapper
model_als_wrapped = ALSWrapper(model_als, user2idx, movie2idx)



In [20]:
print("=== ĐÁNH GIÁ ALS ===")
print(f"Val RMSE: {evaluate_rmse(val_df, model_als_wrapped):.6f}")
print(f"Val NDCG@10: {evaluate_ndcg(val_df, model=model_als_wrapped):.6f}")
print(f"Test NDCG@10: {evaluate_ndcg(test_df, model=model_als_wrapped):.6f}")

p, r, h = evaluate_ranking_metrics(val_df, k=10, threshold=4.0, model=model_als_wrapped)
print(f"Val Precision@10: {p:.4f} | Recall@10: {r:.4f} | HitRate@10: {h:.4f}")

=== ĐÁNH GIÁ ALS ===


IndexError: index 49990 is out of bounds for axis 0 with size 10677

In [10]:
from sklearn.metrics import mean_squared_error

def evaluate_rmse(df_eval,model):
    y_true = []
    y_pred = []

    for _, row in df_eval.iterrows():
        pred = model.predict(row["UserId"], row["MovieId"]).est
        y_true.append(row["Rating"])
        y_pred.append(pred)

    return np.sqrt(mean_squared_error(y_true, y_pred))





In [11]:
from sklearn.metrics import ndcg_score

def evaluate_ndcg(df_eval,model, k=10):
    user_groups = df_eval.groupby("UserId")
    ndcg_list = []

    for user, group in user_groups:
        true_scores = group["Rating"].values.reshape(1, -1)
        pred_scores = np.array([
            model.predict(user, item).est for item in group["MovieId"]
        ]).reshape(1, -1)

        if len(true_scores[0]) > 1:
            ndcg = ndcg_score(true_scores, pred_scores, k=k)
            ndcg_list.append(ndcg)

    return np.mean(ndcg_list)




In [9]:
def evaluate_ranking_metrics(df_eval,model, k=10, threshold=4.0):
    user_groups = df_eval.groupby("UserId")
    
    precisions = []
    recalls = []
    hit_rates = []
    
    for user, group in user_groups:
        item_preds = []
        
        # Dự đoán điểm cho tất cả các item của user trong tập eval
        for _, row in group.iterrows():
            est = model.predict(user, row["MovieId"]).est
            item_preds.append((est, row["Rating"]))
            
        # Sắp xếp giảm dần theo điểm dự đoán (est)
        item_preds.sort(key=lambda x: x[0], reverse=True)
        
        # Lấy Top K item được model đánh giá cao nhất
        top_k = item_preds[:k]
        
        # Đếm số lượng item thực sự relevant (thực tế user đánh giá >= threshold)
        n_rel = sum((true_r >= threshold) for (_, true_r) in item_preds)
        
        # Đếm số lượng item relevant xuất hiện trong Top K (Hits)
        n_rel_and_rec_k = sum((true_r >= threshold) for (_, true_r) in top_k)
        
        # Chỉ tính toán cho các user thực sự có item relevant trong tập eval này
        if n_rel > 0:
            # Precision@K: Tỉ lệ dự đoán đúng trong Top K
            precisions.append(n_rel_and_rec_k / len(top_k))
            
            # Recall@K: Tỉ lệ tìm được so với tổng số item relevant của user đó
            recalls.append(n_rel_and_rec_k / n_rel)
            
            # HitRate@K: Bằng 1 nếu có ít nhất 1 item relevant lọt vào Top K, ngược lại là 0
            hit_rates.append(1 if n_rel_and_rec_k > 0 else 0)
            
    return np.mean(precisions), np.mean(recalls), np.mean(hit_rates)

# Chạy thử trên tập Val và Test




In [14]:

test_precision, test_recall, test_hit_rate = evaluate_ranking_metrics(test_df, k=10, threshold=4.0, model=nmf_model)
print(f"Test Precision@10: {test_precision:.4f} | Test Recall@10: {test_recall:.4f} | Test HitRate@10: {test_hit_rate:.4f}")
print("Test NDCG@10:", evaluate_ndcg(test_df, model=nmf_model))
print("RMSE Test:", evaluate_rmse(test_df, model=nmf_model))


Test Precision@10: 0.6125 | Test Recall@10: 0.8189 | Test HitRate@10: 0.9987
Test NDCG@10: 0.9077098923537067
RMSE Test: 1.6310836544298486


In [17]:
# In thử số lượng item trung bình mỗi user trong val set
print(f"Avg items per user in val_df: {test_df.groupby('UserId').size().mean():.2f}")

# Nếu con số này < 10, thì NDCG@10/Precision@10 có thể không phản ánh đúng khả năng ranking thực

Avg items per user in val_df: 14.58


In [19]:
import random

def evaluate_with_sampling(df_eval, model, train_df, all_movie_ids, k=10, 
                           n_neg_samples=100, threshold=4.0):
    """
    Đánh giá với Positive + Negative Sampling
    
    Parameters:
    -----------
    df_eval: tập val/test cần đánh giá
    model: mô hình đã train
    train_df: tập train (để biết user đã xem phim nào)
    all_movie_ids: danh sách tất cả movie IDs trong hệ thống
    k: top-k để đánh giá
    n_neg_samples: số lượng negative samples cho mỗi user
    threshold: ngưỡng rating để coi là relevant
    """
    
    user_groups = df_eval.groupby("UserId")
    ndcg_list = []
    precisions = []
    recalls = []
    hit_rates = []
    
    for user, group in user_groups:
        # 1. Positive items: các phim trong tập eval (user đã rating)
        positive_items = set(group["MovieId"].values)
        
        # 2. Movies user đã rating trong train (loại khỏi candidate pool)
        rated_in_train = set(train_df[train_df["UserId"] == user]["MovieId"])
        
        # 3. Candidate pool = positive items + negative samples
        # Negative: phim user CHƯA từng xem (không có trong train và không có trong eval)
        unseen_movies = [m for m in all_movie_ids 
                        if m not in rated_in_train and m not in positive_items]
        
        # Sample ngẫu nhiên negative items
        n_neg = min(n_neg_samples, len(unseen_movies))
        negative_items = random.sample(unseen_movies, n_neg)
        
        # Candidate pool cuối cùng
        candidate_items = list(positive_items) + negative_items
        
        # 4. Predict scores cho tất cả candidate items
        pred_scores = []
        true_scores = []
        
        for movie_id in candidate_items:
            pred = model.predict(user, movie_id).est
            pred_scores.append(pred)
            
            # True score: nếu movie có trong eval set → lấy rating thật
            # Nếu không → 0 (negative)
            if movie_id in positive_items:
                rating = group[group["MovieId"] == movie_id]["Rating"].values[0]
                true_scores.append(rating)
            else:
                true_scores.append(0)  # Negative sample
        
        # 5. Tính NDCG@k
        if len(set(true_scores)) > 1:  # Có cả positive và negative
            ndcg = ndcg_score([true_scores], [pred_scores], k=k)
            ndcg_list.append(ndcg)
        
        # 6. Tính Precision, Recall, HitRate@k
        # Sắp xếp theo điểm dự đoán
        item_preds = list(zip(candidate_items, pred_scores, true_scores))
        item_preds.sort(key=lambda x: x[1], reverse=True)  # Sort by pred score
        
        top_k = item_preds[:k]
        
        # Đếm relevant items (rating >= threshold)
        n_rel = sum(1 for (_, _, true_r) in item_preds if true_r >= threshold)
        n_rel_and_rec_k = sum(1 for (_, _, true_r) in top_k if true_r >= threshold)
        
        if n_rel > 0:
            precisions.append(n_rel_and_rec_k / len(top_k))
            recalls.append(n_rel_and_rec_k / n_rel)
            hit_rates.append(1 if n_rel_and_rec_k > 0 else 0)
    
    # 7. Return kết quả
    results = {
        'NDCG@10': np.mean(ndcg_list) if ndcg_list else 0,
        'Precision@10': np.mean(precisions) if precisions else 0,
        'Recall@10': np.mean(recalls) if recalls else 0,
        'HitRate@10': np.mean(hit_rates) if hit_rates else 0
    }
    
    return results

In [23]:
# Chuẩn bị dữ liệu
all_movies = ratings_df['MovieId'].unique()
print(f"Tổng số phim trong hệ thống: {len(all_movies)}")
# Đặt random seed để tái lập kết quả
random.seed(42)
np.random.seed(42)
val_results = evaluate_with_sampling(
    val_df, model_SVD, train_df, all_movies,
    k=10, n_neg_samples=20, threshold=4.0
)

print("\n📊 Kết quả trên Val Set (100 negative samples):")
print(f"  NDCG@10:      {val_results['NDCG@10']:.6f}")
print(f"  Precision@10: {val_results['Precision@10']:.4f}")
print(f"  Recall@10:    {val_results['Recall@10']:.4f}")
print(f"  HitRate@10:   {val_results['HitRate@10']:.4f}")

# Với tập Test
test_results = evaluate_with_sampling(
    test_df, model_SVD, train_df, all_movies,
    k=10, n_neg_samples=20, threshold=4.0
)

print("\n📊 Kết quả trên Test Set (100 negative samples):")
print(f"  NDCG@10:      {test_results['NDCG@10']:.6f}")
print(f"  Precision@10: {test_results['Precision@10']:.4f}")
print(f"  Recall@10:    {test_results['Recall@10']:.4f}")
print(f"  HitRate@10:   {test_results['HitRate@10']:.4f}")

Tổng số phim trong hệ thống: 10677

📊 Kết quả trên Val Set (100 negative samples):
  NDCG@10:      0.598171
  Precision@10: 0.3629
  Recall@10:    0.6370
  HitRate@10:   0.9462

📊 Kết quả trên Test Set (100 negative samples):
  NDCG@10:      0.598776
  Precision@10: 0.3621
  Recall@10:    0.6403
  HitRate@10:   0.9494


In [24]:
# Với tập Test
test_results = evaluate_with_sampling(
    test_df, model_SVDpp, train_df, all_movies,
    k=10, n_neg_samples=20, threshold=4.0
)

print("\n📊 Kết quả model SVD++ trên Test Set (20 negative samples):")
print(f"  NDCG@10:      {test_results['NDCG@10']:.6f}")
print(f"  Precision@10: {test_results['Precision@10']:.4f}")
print(f"  Recall@10:    {test_results['Recall@10']:.4f}")
print(f"  HitRate@10:   {test_results['HitRate@10']:.4f}")

# Với tập Test
test_results = evaluate_with_sampling(
    test_df, nmf_model, train_df, all_movies,
    k=10, n_neg_samples=20, threshold=4.0
)

print("\n📊 Kết quả model NMF trên Test Set (20 negative samples):")
print(f"  NDCG@10:      {test_results['NDCG@10']:.6f}")
print(f"  Precision@10: {test_results['Precision@10']:.4f}")
print(f"  Recall@10:    {test_results['Recall@10']:.4f}")
print(f"  HitRate@10:   {test_results['HitRate@10']:.4f}")


📊 Kết quả model SVD++ trên Test Set (20 negative samples):
  NDCG@10:      0.633676
  Precision@10: 0.3738
  Recall@10:    0.6656
  HitRate@10:   0.9576

📊 Kết quả model NMF trên Test Set (20 negative samples):
  NDCG@10:      0.403262
  Precision@10: 0.4012
  Recall@10:    0.7684
  HitRate@10:   0.9801
